# Precision at fixed candidate budgets (K = 10, 20, 30, 50) -- 8-augmentation templates

**Question.** Same question as `precision_at_k_budgets_14roi.ipynb`, with one change: instead
of matching with a single template per seed click, this notebook matches with an 8-template
augmentation bank per seed (4 rotations x 2 flips x 1 scale) and takes the per-pixel max across
that bank before extraction, NMS and ranking proceed. If the pathologist's candidate list is
capped at 10, 20, 30 or 50 entries, what fraction of what they read is actually a mitotic figure
-- on each of the 14 decision-grade ROIs in `images/extra_valid`, and per tumour domain?
Precision only; `recall@K` is not reported anywhere in this notebook (see the closing summary
for where the recall-family columns that `compare.evaluate_arms` computes as a byproduct end up
instead).

**Scope, fixed for this run:**
- 14 ROIs, `images/extra_valid` (2 per tumour domain x 7 domains).
- 1 seed per ROI, `seed_index = 0` -- same seed draw as `precision_at_k_budgets_14roi.ipynb`,
  since the RNG stream and seed-selection logic are untouched; only what happens to the seed's
  patch downstream (template construction) differs.
- **Single-seed flag.** Every number below comes from 1 click per ROI. Per `experiment.py`'s
  own docstring, single-seed numbers are "illustrative, not a domain ranking... provisional
  until the [multi-seed] sweep is run," and D4's reporting standard is worst-ROI *and* worst
  click, per domain -- Table B here reports worst-ROI only, since one click has no "worst" to
  compare against. Not a defect in this run, just a reminder of how much weight to put on any
  single number below.
- Today's decided defaults (`DECISIONS.md`): `TM_CCOEFF` (D1), no `tissue_mask` (D2),
  `hematoxylin_od` unclipped (D3), ranked by `tm_score` (D5), NMS radius = match radius =
  7.5 um (D7, enforced via `invariants.check_nms_radius`).
- **Template bank: 8 augmentations per seed** -- 4 rotations (0/90/180/270 degrees) x 2 flips
  (`False`, `True`) x 1 scale (`FSConfig(n_angles=4, flips=(False, True))`), the
  `rot90_4angles_2flips` entry of `midog_utils.experiment.AUGMENTATION_VARIANTS`. All 8
  templates share one size (`scales=(1.0,)`), so `tm.fused_response`'s per-pixel max across the
  bank needs no `scale_normalize` correction -- that correction only matters when templates of
  different sizes are compared. This is the only pipeline difference from
  `precision_at_k_budgets_14roi.ipynb`, which uses a single template (no augmentation).
- No z-threshold sweep. Candidates are extracted once per ROI at a permissive deep floor
  (`z = -1.5`, the repo's standing "near-unfiltered" constant), NMS'd at 7.5 um, ranked by
  score, and then `compare.evaluate_arms`'s own budget loop truncates to K = 10/20/30/50. A
  sibling notebook (`find_and_suppress_high_threshold_precision.ipynb`, its Gate 2) already
  showed this is numerically identical to applying any higher threshold first, provided the
  threshold doesn't shrink the pool below K -- which the assertions below confirm holds on
  every ROI at every budget here. Each ROI's own z at rank 10/20/30/50 is reported as a
  diagnostic column, not applied as a filter.


In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils import invariants as inv
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Configuration. D1/D2/D3/D5/D7 per DECISIONS.md. No z-sweep -- one deep-floor extraction
# per ROI, then evaluate_arms's own budget loop does the K = 10/20/30/50 truncation.
# Augmentation: 8 templates per seed (4 rotations x 2 flips x 1 scale) -- the only change
# from precision_at_k_budgets_14roi.ipynb's single-template (n_angles=1, flips=(False,)) run.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0                       # one seed per ROI

CHANNEL = 'hematoxylin_od'           # D3 -- unclipped optical density
METHOD = cv2.TM_CCOEFF               # D1 -- unnormalized, contrast-sensitive
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 10.0               # widened from the sibling notebook's 5.0 -- see cell 2
DEEP_FLOOR_Z = -1.5                  # near-unfiltered extraction floor (repo convention)
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM   # 7.5 um -- D7: NMS radius == match radius
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=4, flips=(False, True), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
assert CFG.n_augmentations == 8, f'expected an 8-template bank, got {CFG.n_augmentations}'
# CFG.self_hit_radius and CFG.max_peaks are vestigial here: both are only read inside
# fs.find_and_suppress() (find_and_suppress.py:141,156), which this notebook's inline
# run_roi() never calls. Self-hit removal and the peak cap actually in effect are the
# module-level SELF_HIT_RADIUS / MAX_PEAKS constants above/below, read directly by
# suppress() / extract_peaks().
BORDER = CFG.patch_size // 2         # 36
OTSU_WINDOW = tm.BASE_SIZE           # 51
LARGEST_CC_KW = dict(min_area=50, max_area_frac=0.85, min_solidity=0.5)

BUDGETS = (10, 20, 30, 50)           # the fixed candidate-list lengths under test

OUT_PER_ROI = '../results/precision_at_k_14roi_8aug_per_roi.csv'
OUT_BY_DOMAIN = '../results/precision_at_k_14roi_8aug_by_domain.csv'
OUT_RAW = '../results/precision_at_k_14roi_8aug_raw.csv'
OUT_VERIF = '../results/precision_at_k_14roi_8aug_verification.csv'

print(f'budgets {BUDGETS} x 14 ROIs x 1 click (seed_index={SEED_INDEX})')
print(f'template bank: {CFG.n_augmentations} augmentations per seed '
      f'({len(CFG.scales)} scale x {CFG.n_angles} angles x {len(CFG.flips)} flips)')
print(f'NMS radius = match radius = {NMS_RADIUS_UM} um (D7)')


budgets (10, 20, 30, 50) x 14 ROIs x 1 click (seed_index=0)
template bank: 8 augmentations per seed (1 scale x 4 angles x 2 flips)
NMS radius = match radius = 7.5 um (D7)


## The pipeline

Copied from `precision_at_k_budgets_14roi.ipynb`, which copied it from
`find_and_suppress_high_threshold_precision.ipynb`, which copied it from
`recall_workload_ledger.py` -- the seed draw, template-tightening and NMS+self-hit helpers are
kept inline rather than promoted to `midog_utils`, for the same reason that notebook gives: the
bbox-tightening rule was probed and explicitly not adopted as production logic.

`largest_cc_box` below (the inline seed-sizing helper) picks the largest Otsu component in the
click's crop without checking the click itself falls inside it -- unlike production's
`seed_selection.tighten_box_otsu`, which refuses that fallback by design. This is a pre-existing
simplification, not introduced by this notebook: `Research Logs/2026-09-08-f1-independent-audit.md`
found it identical to `tighten_box_otsu` on 35/35 seeds it checked. Seed draw and sizing happen on
the unrotated click patch before any augmentation is applied, so that equivalence doesn't depend
on the augmentation config -- it just hasn't been independently re-checked for these particular
14 seeds.

**One deliberate change from the single-template version, beyond the augmentation config
itself:** `suppress()`'s self-hit filter is widened from 5.0px to 10.0px (`SELF_HIT_RADIUS`, set
in the config cell above). Under a single template, the seed's own self-correlation is provably
the fused map's global maximum, so NMS (greedy, descending score order) suppresses everything
within one match radius of it before the self-hit filter ever runs -- 5.0px was already headroom
on top of an already-empty annulus. Under the 8-augmentation bank that guarantee doesn't
automatically carry over: a rotated/flipped template can score marginally higher a few pixels
off from the exact click, which left the seed's own detection sitting 6.08px away on 403.tiff --
just past the original 5.0px window, confirmed to still be the seed's own signature (nothing
else exists in that ROI's 5-33px band after NMS).

**Why 10.0px, and not the match radius (~30-33px):** widening all the way to match radius was
tried and reverted. `find_and_suppress.py`'s own design notes explain why it deliberately keeps
the self-hit radius tight rather than using the match radius: the minimum spacing between two
*real* MIDOG++ annotations anywhere in the dataset is 26.2px (403.tiff; 245.tiff is 26.6px) --
below the ~30px match radius -- so a match-radius self-hit filter can delete a legitimate
detection of a neighbouring ground-truth object, not just the seed's own echo. (On this run's
specific 14 seeds that risk didn't fire -- independently checked, the nearest other mitotic
figure to any seed is 61.98px away, on 094.tiff -- but that's specific to this seed draw, not
guaranteed by construction, so a match-radius filter isn't a safe default.) 10.0px covers the
observed 6.08px displacement with margin while staying well clear of 26.2px, and leaves the
"Checks before any table" section below as a real, non-tautological check: its filter radius
(10.0px) and the check's radius (match radius, ~30px) are different values, so the check can
still fail and would catch a future case this margin doesn't cover.


In [2]:
def _odd_local(n, minimum=5):
    n = int(round(n))
    if n % 2 == 0:
        n += 1
    return max(minimum, n)


def largest_cc_box(patch, min_area=50, max_area_frac=0.85, min_solidity=0.5):
    '''Otsu-threshold `patch`; return the bbox of its largest connected component by area.

    (y0, y1, x0, x1), patch-local, half-open, or None when the component fails a gate.
    '''
    if patch.ndim != 2:
        raise ValueError(f'largest_cc_box needs a single-channel patch, got {patch.shape}')
    u8 = cv2.normalize(patch.astype(np.float32), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, binary = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    regions = regionprops(label(binary, connectivity=2))
    if not regions:
        return None
    largest = max(regions, key=lambda r: r.area)
    if largest.area < min_area or largest.area > max_area_frac * patch.size:
        return None
    if largest.solidity < min_solidity:
        return None
    y0, x0, y1, x1 = largest.bbox
    return int(y0), int(y1), int(x0), int(x1)


def draw_seed_with_retry(pool, rng, check_fn):
    '''Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.'''
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, seed_xy):
    '''NMS at `radius`, then drop the seed's own self-correlation. Returns (centers, scores).

    Self-hit removal uses the fixed SELF_HIT_RADIUS (10.0px here, vs. the sibling notebook's
    5.0px) -- not `radius` (== match_radius, D7). Widening this filter all the way to match
    radius was tried and reverted: `find_and_suppress.py` documents that the minimum spacing
    between two real MIDOG++ annotations anywhere in the dataset is 26.2px, below the ~30px
    match radius, so a match-radius self-hit filter risks deleting a legitimate neighbouring
    detection, not just the seed's own echo. See cell 2 (`## The pipeline`) for the full
    reasoning and the empirical case (403.tiff) that motivated widening past 5.0px at all.
    '''
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - seed_xy[0], c[:, 1] - seed_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## The run

Per ROI: eight `matchTemplate` passes -- one per augmented template (4 rotations x 2 flips x
1 scale) -- fused into a single response map by per-pixel max in `tm.fused_response`, then one
extraction at the deep floor, one NMS at 7.5 um. matchTemplate cost scales with template count,
but per-ROI fixed costs (image load, channel conversion, peak extraction, NMS) don't scale with
template count and dominate the wall-clock budget here: across three full executions of this
notebook while developing it, total runtime ranged 169-267s for 14 ROIs (vs. the original
single-template notebook's one measured run at 193s) -- close to parity, well short of an 8x
multiplier, but with enough run-to-run variance on this machine that a single-point ratio isn't
worth quoting as fact. No re-run per budget -- `compare.evaluate_arms` computes `tp_at_budget` /
`budget_delivered` for every K in `BUDGETS` from that single ranked list.


In [3]:
def run_roi(fn, image_id, domain, anns):
    t0 = time.time()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb
    gc.collect()

    # --- the click -------------------------------------------------------------------
    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        p = tm.read_padded_patch(gray_inv, float(row['cx']), float(row['cy']), OTSU_WINDOW)
        return None if p is None else largest_cc_box(p, **LARGEST_CC_KW)

    seed, seed_box, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    y0, y1, x0, x1 = seed_box
    base_size = _odd_local(max(y1 - y0, x1 - x0))
    seed_xy = (float(seed['cx']), float(seed['cy']))
    seed_ann_id = int(seed['ann_id'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    del gray_inv
    gc.collect()

    # --- one match, one deep-floor extraction, one NMS -- no z-sweep --------------------
    patch = tm.read_padded_patch(hem, *seed_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    del hem_p, fused_p, valid_p, hem
    gc.collect()

    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    n_peaks = len(centers)
    assert n_peaks < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    c, s = suppress(centers, scores, nms_radius, seed_xy)
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})

    # Seed annulus: under the 8-augmentation bank, NMS alone does not reliably empty this
    # (unlike the single-template sibling notebook) -- the fixed SELF_HIT_RADIUS filter in
    # suppress() is what closes the gap. This must count zero here; see cell 2 / 790a2936
    # ('Checks before any table') for why that is not a tautology at SELF_HIT_RADIUS < match
    # radius.
    d_seed = np.hypot(pool['cx'] - seed_xy[0], pool['cy'] - seed_xy[1])
    n_near_seed = int((d_seed <= match_radius).sum())

    arm = cp.Arm('tm_score_deep_floor', (lambda d=pool: d), rank_key='score',
                 seeded=True, z=DEEP_FLOOR_Z, z_dependent=True,
                 nms_radius=nms_radius, caps=(MAX_PEAKS,))
    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id,
               seed_ann_id=seed_ann_id, base_size=base_size,
               map_median=round(float(med), 5), mad_scale=round(float(mad), 5), mpp=mpp)
    checks = []
    out = cp.evaluate_arms([arm], gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                            budgets=BUDGETS, context=ctx, checks=checks)

    z_at_rank = {}
    for k in BUDGETS:
        z_at_rank[k] = (float(pool['score'].iloc[k - 1] - med) / mad) if len(pool) >= k else np.nan
    out['z_at_rank'] = out['budget'].map(z_at_rank)

    checks.append(dict(check='seed_annulus_empty', label=fn, n_near_seed=n_near_seed,
                       match_radius_px=round(match_radius, 3), passed=bool(n_near_seed == 0)))

    meta = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                n_retries=n_retries, contested_seed_tier=bool(flagged), base_size=base_size,
                mpp=mpp, roi_h=int(H), roi_w=int(W), pad_px=PAD,
                match_radius_px=match_radius, nms_radius_px=nms_radius,
                map_median=float(med), mad_scale=float(mad), n_gt_mitotic=n_gt,
                n_detections=len(pool), n_near_seed_annulus=n_near_seed,
                t_total_s=round(time.time() - t0, 1))
    del fused, valid, templates, patch, centers, scores, c, s
    gc.collect()
    print(f"[{fn}] {domain:32s} base={base_size:2d} n_detections={len(pool):6d} "
          f"n_gt={n_gt:3d} [{meta['t_total_s']:.0f}s]", flush=True)
    return out, pd.DataFrame(checks), meta

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
missing = sorted(set(files) - set(meta_ix.index))
assert not missing, f'.tiff on disk absent from the annotation DB: {missing}'
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

t_run = time.time()
out_frames, check_frames, roi_meta = [], [], []
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    out, checks, m = run_roi(fn, image_id, domain, annotations)
    out_frames.append(out)
    check_frames.append(checks)
    roi_meta.append(m)
    gc.collect()

RAW = pd.concat(out_frames, ignore_index=True)
VERIF = pd.concat(check_frames, ignore_index=True)
ROI = pd.DataFrame(roi_meta).set_index('file_name')

print(f'\n{len(files)} ROIs x {len(BUDGETS)} budgets = {len(RAW)} rows in {time.time() - t_run:.0f}s')
ROI[['tumor_type', 'seed_ann_id', 'base_size', 'n_gt_mitotic', 'n_detections',
     'match_radius_px', 'nms_radius_px', 'map_median', 'mad_scale', 't_total_s']].round(3)

[013.tiff] human breast cancer              base=31 n_detections= 17045 n_gt= 17 [14s]
[094.tiff] human breast cancer              base=25 n_detections= 18109 n_gt= 81 [12s]
[201.tiff] canine lung cancer               base=51 n_detections= 15536 n_gt= 17 [12s]
[233.tiff] canine lung cancer               base=25 n_detections= 17489 n_gt= 17 [10s]
[245.tiff] canine lymphosarcoma             base=47 n_detections= 19191 n_gt= 89 [12s]
[246.tiff] canine lymphosarcoma             base=41 n_detections= 17573 n_gt=115 [11s]
[300.tiff] canine cutaneous mast cell tumor base=45 n_detections= 17792 n_gt=180 [11s]
[301.tiff] canine cutaneous mast cell tumor base=41 n_detections= 17805 n_gt=217 [10s]
[402.tiff] human neuroendocrine tumor       base=29 n_detections= 17169 n_gt=104 [14s]
[403.tiff] human neuroendocrine tumor       base=33 n_detections= 17236 n_gt= 52 [15s]
[459.tiff] canine soft tissue sarcoma       base=33 n_detections= 17827 n_gt=130 [10s]
[460.tiff] canine soft tissue sarcoma      

,tumor_type,seed_ann_id,base_size,n_gt_mitotic,n_detections,match_radius_px,nms_radius_px,map_median,mad_scale,t_total_s
file_name,,,,,,,,,,
013.tiff,human breast cancer,254,31,17,17045,33.139,33.139,0.173,0.180,13.6
094.tiff,human breast cancer,2512,25,81,18109,32.630,32.630,0.038,0.052,11.8
201.tiff,canine lung cancer,4457,51,17,15536,30.222,30.222,0.401,0.581,11.5
233.tiff,canine lung cancer,5761,25,17,17489,30.222,30.222,0.037,0.140,9.6
245.tiff,canine lymphosarcoma,6274,47,89,19191,30.222,30.222,0.270,0.150,11.6
246.tiff,canine lymphosarcoma,6548,41,115,17573,30.222,30.222,0.133,0.154,10.6
300.tiff,canine cutaneous mast cell tumor,14581,45,180,17792,29.609,29.609,0.220,0.393,10.6
301.tiff,canine cutaneous mast cell tumor,14969,41,217,17805,29.609,29.609,0.165,0.194,10.1
402.tiff,human neuroendocrine tumor,20254,29,104,17169,33.139,33.139,0.669,0.874,14.0


## Checks before any table

Three things must hold for the deep-floor-then-truncate shortcut to be valid, and for the D7
NMS-radius invariant to be more than documentation:

1. Every ROI's deep-floor pool clears the largest budget (K = 50) -- otherwise a
   "precision@50" would silently be computed over fewer than 50 candidates.
2. `budget_delivered == budget` for every row -- the direct restatement of (1) at every K, not
   just the largest.
3. No surviving candidate lies within one match radius of the seed's own (removed) annotation.
   Under a single template this is guaranteed structurally: the self-correlation is the response
   map's global maximum, so NMS keeps it first and suppresses everything within one match radius
   before self-hit removal ever runs. Under this notebook's 8-augmentation bank that guarantee
   doesn't automatically hold (see cell 2), so this check is not a tautology here -- its radius
   (match radius, ~30px) is deliberately different from `suppress()`'s self-hit filter radius
   (a fixed 10.0px, not widened to match radius -- see cell 2 for why). NMS and the small
   self-hit filter have to jointly clear the whole annulus for this to pass, and this check is
   what verifies they actually did, on every ROI -- not just 403.tiff, the one that motivated
   widening the filter past 5.0px in the first place.

`invariants.check_nms_radius` (D7) and `invariants.check_no_cap` were already asserted per ROI
inside `evaluate_arms`, above, and are already sitting in `VERIF`.


In [5]:
assert (ROI['n_detections'] >= max(BUDGETS)).all(), \
    'a deep-floor pool did not clear the largest budget -- see ROI[\'n_detections\']'
assert (RAW['budget_delivered'] == RAW['budget']).all(), \
    'a budget row was starved -- the deep floor did not have enough survivors somewhere'
assert (ROI['n_near_seed_annulus'] == 0).all(), \
    'a candidate survived inside the removed seed\'s match radius -- see n_near_seed_annulus'

VERIF = pd.concat([VERIF, pd.DataFrame([
    dict(check='deep_pool_covers_max_budget', label='ALL',
         passed=bool((ROI['n_detections'] >= max(BUDGETS)).all())),
    dict(check='no_starvation_any_budget', label='ALL',
         passed=bool((RAW['budget_delivered'] == RAW['budget']).all())),
])], ignore_index=True)
VERIF.to_csv(OUT_VERIF, index=False)
print(f'{len(VERIF)} verification records -> {OUT_VERIF}')
print(f"  passed: {int(VERIF['passed'].sum())} / {len(VERIF)}")
assert VERIF['passed'].all(), 'a check failed -- read the verification CSV before any table below'

44 verification records -> ../results/precision_at_k_14roi_8aug_verification.csv
  passed: 44 / 44


## Table A -- precision per ROI, at each budget

One row per ROI (14 total), sorted by domain. `n_gt_mitotic` and `n_detections` are descriptive
context, not a recall metric. `z_at_rank_K` is the per-ROI/per-domain robust-z value that
candidate K happens to sit at -- reported because it visibly varies (as flagged going in),
never applied as a threshold.

In [6]:
RAW['precision_at_budget'] = RAW['tp_at_budget'] / RAW['budget_delivered']

order = ROI[['tumor_type']].reset_index().sort_values(['tumor_type', 'file_name'])

rows_a = []
for _, o in order.iterrows():
    fn = o['file_name']
    r = dict(file_name=fn, domain=o['tumor_type'],
             n_gt_mitotic=int(ROI.loc[fn, 'n_gt_mitotic']),
             n_detections=int(ROI.loc[fn, 'n_detections']))
    sub = RAW[RAW['file_name'] == fn].set_index('budget')
    for k in BUDGETS:
        r[f'z_at_rank_{k}'] = round(float(sub.loc[k, 'z_at_rank']), 3)
        r[f'budget_delivered_{k}'] = int(sub.loc[k, 'budget_delivered'])
        r[f'tp_at_{k}'] = int(sub.loc[k, 'tp_at_budget'])
        r[f'precision_at_{k}'] = round(float(sub.loc[k, 'precision_at_budget']), 4)
    rows_a.append(r)

TABLE_A = pd.DataFrame(rows_a)
TABLE_A.to_csv(OUT_PER_ROI, index=False)
print(f'-> {OUT_PER_ROI}  ({len(TABLE_A)} rows)')
TABLE_A

-> ../results/precision_at_k_14roi_8aug_per_roi.csv  (14 rows)


,file_name,domain,n_gt_mitotic,n_detections,z_at_rank_10,budget_delivered_10,tp_at_10,precision_at_10,z_at_rank_20,budget_delivered_20,tp_at_20,precision_at_20,z_at_rank_30,budget_delivered_30,tp_at_30,precision_at_30,z_at_rank_50,budget_delivered_50,tp_at_50,precision_at_50
0,300.tiff,canine cutaneous mast cell tumor,180,17792,8.032,10,8,0.8,7.289,20,14,0.70,7.124,30,23,0.7667,6.775,50,36,0.72
1,301.tiff,canine cutaneous mast cell tumor,217,17805,7.023,10,7,0.7,6.816,20,16,0.80,6.648,30,25,0.8333,6.388,50,41,0.82
2,201.tiff,canine lung cancer,17,15536,7.009,10,3,0.3,6.548,20,6,0.30,6.206,30,7,0.2333,5.844,50,10,0.20
3,233.tiff,canine lung cancer,17,17489,11.403,10,5,0.5,10.613,20,9,0.45,10.299,30,9,0.3000,9.962,50,9,0.18
4,245.tiff,canine lymphosarcoma,89,19191,6.854,10,0,0.0,6.249,20,2,0.10,6.101,30,2,0.0667,5.869,50,3,0.06
5,246.tiff,canine lymphosarcoma,115,17573,11.730,10,10,1.0,10.694,20,19,0.95,9.968,30,29,0.9667,8.919,50,42,0.84
6,459.tiff,canine soft tissue sarcoma,130,17827,7.845,10,7,0.7,7.425,20,11,0.55,7.029,30,15,0.5000,6.682,50,25,0.50
7,460.tiff,canine soft tissue sarcoma,35,13729,7.196,10,9,0.9,6.784,20,12,0.60,6.425,30,16,0.5333,6.042,50,18,0.36
8,013.tiff,human breast cancer,17,17045,15.195,10,5,0.5,13.457,20,7,0.35,12.973,30,10,0.3333,12.048,50,13,0.26
9,094.tiff,human breast cancer,81,18109,18.497,10,5,0.5,17.562,20,12,0.60,16.979,30,18,0.6000,15.707,50,24,0.48


## Table B -- precision per domain, at each budget

7 domains x 4 budgets = 28 rows. `precision_pooled = sum(tp_at_budget) / sum(budget_delivered)`
across that domain's 2 ROIs -- algebraically identical to the simple mean of the two ROIs'
`precision_at_K` here, because the checks above guarantee `budget_delivered == K` for both. The
worst-ROI columns name the weaker of the two ROIs per domain/budget, so a bad cell can't hide
behind the pooled number.

In [7]:
# RAW already carries 'tumor_type' (it was passed through `context` into evaluate_arms),
# so this groups it directly rather than re-merging against ROI and colliding column names.
rows_b = []
for (domain, k), g in RAW.groupby(['tumor_type', 'budget']):
    tp_sum = int(g['tp_at_budget'].sum())
    delivered_sum = int(g['budget_delivered'].sum())
    worst = g.loc[g['precision_at_budget'].idxmin()]
    rows_b.append(dict(
        domain=domain, n_roi=len(g), K=int(k),
        tp_sum=tp_sum, delivered_sum=delivered_sum,
        precision_pooled=round(tp_sum / delivered_sum, 4) if delivered_sum else np.nan,
        precision_worst_roi=round(float(worst['precision_at_budget']), 4),
        worst_roi_file=str(worst['file_name']),
    ))

TABLE_B = pd.DataFrame(rows_b).sort_values(['domain', 'K']).reset_index(drop=True)
TABLE_B.to_csv(OUT_BY_DOMAIN, index=False)
print(f'-> {OUT_BY_DOMAIN}  ({len(TABLE_B)} rows = 7 domains x {len(BUDGETS)} budgets)')
TABLE_B

-> ../results/precision_at_k_14roi_8aug_by_domain.csv  (28 rows = 7 domains x 4 budgets)


,domain,n_roi,K,tp_sum,delivered_sum,precision_pooled,precision_worst_roi,worst_roi_file
0,canine cutaneous mast cell tumor,2,10,15,20,0.7500,0.7000,301.tiff
1,canine cutaneous mast cell tumor,2,20,30,40,0.7500,0.7000,300.tiff
2,canine cutaneous mast cell tumor,2,30,48,60,0.8000,0.7667,300.tiff
3,canine cutaneous mast cell tumor,2,50,77,100,0.7700,0.7200,300.tiff
4,canine lung cancer,2,10,8,20,0.4000,0.3000,201.tiff
5,canine lung cancer,2,20,15,40,0.3750,0.3000,201.tiff
6,canine lung cancer,2,30,16,60,0.2667,0.2333,201.tiff
7,canine lung cancer,2,50,19,100,0.1900,0.1800,233.tiff
8,canine lymphosarcoma,2,10,10,20,0.5000,0.0000,245.tiff
9,canine lymphosarcoma,2,20,21,40,0.5250,0.1000,245.tiff


## Closing summary

`precision_at_K` (Table A) and `precision_pooled` / `precision_worst_roi` (Table B) are the
deliverable. `compare.evaluate_arms` computes `recall_at_budget`, `full_list_recall` and
`read_50..read_100` as an unavoidable byproduct of reusing that harness -- it's what the module
was built for -- but those columns exist only in `results/precision_at_k_14roi_8aug_raw.csv`,
for provenance. They are never selected into Table A or Table B and are not part of this
analysis.


In [8]:
RAW.to_csv(OUT_RAW, index=False)
print(f'-> {OUT_RAW}  ({len(RAW)} rows; recall-family columns retained here only, for provenance)')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

-> ../results/precision_at_k_14roi_8aug_raw.csv  (56 rows; recall-family columns retained here only, for provenance)

notebook ran in 169s
